In [1]:
from rich.console import Console


console = Console()


def showtext(text):
    try:
        console.print(text)
    except Exception as e:
        print(f"Console print failed: {e}\n")
        print(text)

In [2]:
todos = []  # 需要完成的任务
completed = []  # 标记任务完成状态

In [3]:
def get_todo_report():
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:  # 已经完成
            result += f"待完成 #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"待完成 #{index + 1}: {todo}\n"
    showtext(result)
    return result

In [4]:
# 扩展 todos, completed(默认 False)
def create_todos(descriptions: list[str]):
    todos.extend(descriptions)
    completed.extend([False] * len(todos))  # 默认状态全为 false
    return get_todo_report()

In [5]:
def mark_complete(index: int, complete_notes: str):  # index 是 complete_notes 的 index
    i = index - 1  # index 起始编号为 1
    if 0 <= i < len(todos):
        completed[i] = True  # 更新状态
        showtext(complete_notes)
        return get_todo_report()
    else:
        return f"索引{index}没有对应的待完成事项"

In [6]:
todos.clear()
completed.clear()

In [14]:
create_todos(["买杂货", "完成其它实验", "吃苹果"])

待完成 #1: 买杂货
待完成 #2: 完成其它实验
待完成 #3: 吃苹果

'待完成 #1: 买杂货\n待完成 #2: 完成其它实验\n待完成 #3: 吃苹果\n'

In [8]:
# mark_complete(1, "买")

In [9]:
# function 元数据
# "description" 是对函数的描述
# "properties" 下的"descriptions" 是参数名称
# "title" 的作用类似 lable, 给参数加一个标签
create_todos_json = {
    "name": "create_todos",
    "description": "添加新待办事项并返回包含所有待办状态的完整报告",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                "type": "list",
                "items": {"type": "string"},
                "title": "Descriptions",
            }
        },
        "required": ["descriptions"],
        "additionalProperties": False,
    },
}

In [10]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "将编号为 index 的代办项标记为 True 并返回包含所有待办状态的完整报告",
    "parameters": {
        "type": "object",
        "proerties": {
            "index": {"type": "integer", "title": "Index"},
            "complete_notes": {"type": "string", "title": "CompleteNotes"},
        },
        "required": ["Index", "complete_notes"],
        "additionalProperties": False,
    },
}

In [11]:
tools = [
    {"name": "function", "function": create_todos_json},
    {"name": "function", "function": mark_complete_json},
]

In [12]:
from openai.types.chat import (
    ParsedChatCompletion,
    ChatCompletionMessageToolCallUnion,
    ChatCompletionMessageFunctionToolCall,
    ChatCompletionMessage,
)
import json

In [ ]:
def handle_tool_json(tool_calls: list[ChatCompletionMessageToolCallUnion]):
    results = []
    for tool_call in tool_calls:
        if (
            isinstance(tool_call, ChatCompletionMessageFunctionToolCall)
            and tool_call.function is not None
        ):
            tool_name = tool_call.function.name
            parameters = json.loads(tool_call.function.arguments)

            tool = globals().get(tool_name)
            todo_result = tool(**parameters) if tool else {}
            results.append(
                {
                    "role": "tool",
                    "content": json.dumps(todo_result),
                    "tool_call_id": tool_call.id,
                }
            )
    return results